In [ ]:
# Setup: connect Google Drive and open the project folder (works in Google Colab)
import os, sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/NovelMate_ML_Project"
except ImportError:                      # running on your own computer instead of Colab
    PROJECT_DIR = "."
if not Path(PROJECT_DIR).exists():
    raise FileNotFoundError("Upload the NovelMate_ML_Project folder to My Drive first (see README.md).")
os.chdir(PROJECT_DIR)
sys.path.insert(0, os.getcwd())

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report, f1_score,
                             mean_absolute_error, mean_squared_error, r2_score)

from ml_utils import *          # paths, FEATURES, build_pipeline, split_data, register_model ...

In [ ]:
df = load_clean_data()
X_train, X_test, yr_train, yr_test, yc_train, yc_test = split_data(df)
print("Training novels:", len(X_train), "| Test novels:", len(X_test))
print("Features:", FEATURES)

In [ ]:
from sklearn.svm import SVR

model = build_pipeline(SVR())
model.fit(X_train, yr_train)

pred = np.clip(model.predict(X_test), 0, 5)          # ratings only exist between 0 and 5
r2 = r2_score(yr_test, pred)
mae = mean_absolute_error(yr_test, pred)
rmse = np.sqrt(mean_squared_error(yr_test, pred))
baseline = mean_absolute_error(yr_test, np.full(len(yr_test), yr_train.mean()))
print(f"R2   : {r2:.4f}   (1 = perfect, 0 = no better than predicting the mean)")
print(f"MAE  : {mae:.4f}   (baseline that always predicts the mean: {baseline:.4f})")
print(f"RMSE : {rmse:.4f}")

In [ ]:
plt.figure(figsize=(5, 5))
plt.scatter(yr_test, pred, s=8, alpha=0.4, color="#8b3a2f")
plt.plot([1.5, 5], [1.5, 5], "k--", label="perfect prediction")
plt.xlabel("Actual average rating"); plt.ylabel("Predicted average rating")
plt.title("Actual vs predicted (test set)"); plt.legend(); plt.show()

In [ ]:
register_model("regression", "svr", "Support Vector Regression (SVR)", model,
               {"r2": r2, "mae": mae, "rmse": rmse})